<a href="https://colab.research.google.com/github/TimofeyProtasov/diploma/blob/main/days/work_1904.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q triton trl evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 54.3 MB/s eta 0:00:00


In [2]:
import re
import torch
import random
import numpy as np
import pandas as pd
import time
import evaluate
from typing import Optional
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from dataclasses import dataclass, field
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
from trl import SFTTrainer, SFTConfig
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    PeftConfig
)
from tqdm import tqdm
from functools import wraps
from torch.utils.data import DataLoader

In [3]:
def measure_metrics(metric_name: str):
    """
    Декоратор для измерения времени выполнения и пиковой памяти GPU метода.
    Сохраняет результаты в self.metrics с ключами:
    - f"{metric_name}_time_min"
    - f"{metric_name}_peak_memory_gb"
    """
    def decorator(func):
        @wraps(func)
        def wrapper(self, *args, **kwargs):
            # Сбрасываем статистику пиковой памяти, если доступна CUDA
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()

            start_time = time.time()

            # Выполняем сам метод
            result = func(self, *args, **kwargs)

            elapsed_min = (time.time() - start_time) / 60.0
            self.metrics[f"{metric_name}_time_min"] = round(elapsed_min, 2)

            if torch.cuda.is_available():
                peak_memory_gb = torch.cuda.max_memory_allocated() / (1024**3)
                self.metrics[f"{metric_name}_peak_memory_gb"] = round(peak_memory_gb, 2)
            else:
                self.metrics[f"{metric_name}_peak_memory_gb"] = 0.0

            return result
        return wrapper
    return decorator

In [4]:
@dataclass
class RAGExperiment:
    dataset_name: str = 'phatvo/hotpotqa-raft'
    oracle_presence_ratio: float = 1.0
    train_size: int = 10
    test_size: int = 2
    model_name: str = 'Qwen/Qwen2.5-0.5B-Instruct'
    peft_config: Optional[PeftConfig] = field(default_factory=lambda: LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    ))
    num_train_epochs: int = 3
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 2
    learning_rate: float = 5e-5
    warmup_steps: int = 5
    seed: int = 1


    def __post_init__(self):
        random.seed(self.seed)
        self.squad_metric = evaluate.load("squad")
        self.metrics = {}


    def prepare_data(self):
        dataset: DatasetDict = load_dataset('phatvo/hotpotqa-raft')


        def presence_oracle(example):
            return example['oracle_context'] in example['instruction']


        dataset_with_oracle: Dataset = dataset['train'].filter(lambda ex: presence_oracle(ex))
        dataset_without_oracle: Dataset = dataset['train'].filter(lambda ex: not presence_oracle(ex))

        self.sample_size = self.train_size + self.test_size

        num_with: int = round(self.sample_size * self.oracle_presence_ratio)
        num_without: int = round(self.sample_size * (1 - self.oracle_presence_ratio))

        if (num_with > len(dataset_with_oracle)) or (num_without > len(dataset_without_oracle)):
            raise ValueError("not enough samples in dataset")

        indices_with: list = random.sample(range(len(dataset_with_oracle)), num_with)
        size_train_with = round(self.train_size * self.oracle_presence_ratio)

        indices_without: list = random.sample(range(len(dataset_without_oracle)), num_without)
        size_train_without = round(self.train_size * (1 - self.oracle_presence_ratio))

        idx_train_with_oracle = indices_with[:size_train_with]
        idx_test_with_oracle = indices_with[size_train_with:num_with]

        idx_train_without_oracle = indices_without[:size_train_without]
        idx_test_without_oracle = indices_without[size_train_without:num_without]

        def get_shuffle_dataset(ids_with, ids_without):
            parts = []
            if ids_with:
                parts.append(dataset_with_oracle.select(ids_with))
            if ids_without:
                parts.append(dataset_without_oracle.select(ids_without))
            if parts:
                return concatenate_datasets(parts).shuffle(seed=self.seed)
            else:
                raise ValueError('empty sample')


        self.train_dataset: Dataset = get_shuffle_dataset(idx_train_with_oracle, idx_train_without_oracle)
        self.test_dataset: Dataset = get_shuffle_dataset(idx_test_with_oracle, idx_test_without_oracle)


    def prepare_model(self):
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            dtype=torch.bfloat16,
            device_map='auto'
        )


    def prepare_tokenizer(self):
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name
        )
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token


    def format_data(self, dataset):
        """Форматируем данные в prompt/completion для SFTTrainer."""
        def format_sft_example(example):
            prompt_messages = [{"role": "user", "content": example["instruction"]}]
            prompt = self.tokenizer.apply_chat_template(
                prompt_messages,
                tokenize=False,
                add_generation_prompt=True
            )
            completion = example["cot_answer"]
            return {"prompt": prompt, "completion": completion}

        return dataset.map(
            format_sft_example,
            remove_columns=dataset.column_names
        )


    def add_peft_if_exist(self):
        self.model = get_peft_model(self.model, self.peft_config) if self.peft_config else self.model
        self.metrics["trainable_params_m"] = self.model.num_parameters(only_trainable=True) / 1e6


    def prepare_training(self):
        self.training_args = SFTConfig(
            output_dir="./qwen-raft-lora",
            num_train_epochs=self.num_train_epochs,
            per_device_train_batch_size=self.per_device_train_batch_size,
            gradient_accumulation_steps=self.gradient_accumulation_steps,
            learning_rate=self.learning_rate,
            warmup_steps=self.warmup_steps,
            logging_steps=10,
            save_steps=100,
            save_total_limit=2,
            bf16=True,
            report_to="none",
            remove_unused_columns=False,

            max_length=1280,
            packing=False,
            completion_only_loss=True,
        )

        self.trainer = SFTTrainer(
            model=self.model,
            args=self.training_args,
            train_dataset=self.formatted_train_dataset,
            processing_class=self.tokenizer,
        )

    @measure_metrics("train")
    def train(self):
        self.trainer.train()


    def extract_answer(self, text: str) -> Optional[str]:
        """
        Извлекает ответ после <ANSWER>:.
        Возвращает None, если маркер отсутствует.
        """
        match = re.search(r"<ANSWER>:\s*(.*?)(?:\n|$)", text, re.DOTALL)
        if match:
            return match.group(1).strip()
        return None


    @measure_metrics("evaluate_f1")
    def evaluate_f1(self):
        """
        Вычисляет среднюю F1-меру по тестовому набору, используя метрику SQuAD.
        Если в сгенерированном тексте нет тега <ANSWER>:, F1 = 0.
        """
        f1_scores = []

        for i, example in enumerate(tqdm(self.formatted_test_dataset, desc="Evaluating F1")):
            prompt = example["prompt"]
            true_completion = example["completion"]
            true_answer = self.extract_answer(true_completion)
            if true_answer is None:
                continue

            inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

            with torch.no_grad():
                generate_kwargs = {
                    **inputs,
                    "max_new_tokens": 1024,
                    "do_sample": False,
                    "pad_token_id": self.tokenizer.eos_token_id,
                    "eos_token_id": self.tokenizer.eos_token_id,
                    "temperature": None,
                    "top_p": None,
                    "top_k": None,
                }
                outputs = self.model.generate(**generate_kwargs)

            full_output = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            generated_part = full_output[len(prompt):].strip()
            pred_answer = self.extract_answer(generated_part)

            if pred_answer is None:
                f1 = 0.0
            else:
                pred_dict = {"prediction_text": pred_answer, "id": str(i)}
                ref_dict = {"answers": {"answer_start": [0], "text": [true_answer]}, "id": str(i)}
                result = self.squad_metric.compute(predictions=[pred_dict], references=[ref_dict])
                f1 = result["f1"] / 100.0

            f1_scores.append(f1)

        avg_f1 = np.mean(f1_scores) if f1_scores else 0.0

        self.metrics["f1"] = round(float(avg_f1), 2)


    @measure_metrics("evaluate_f1_batch_2")
    def evaluate_f1_batch_2(self, batch_size: int = 2):
        """
        Ба́тчевая версия evaluate_f1 для ускорения генерации.
        """
        original_padding_side = self.tokenizer.padding_side
        self.tokenizer.padding_side = "left"

        def collate_fn(batch):
            prompts = [item["prompt"] for item in batch]
            completions = [item["completion"] for item in batch]
            true_answers = [self.extract_answer(c) for c in completions]

            tokenized = self.tokenizer(
                prompts,
                padding=True,
                truncation=True,
                max_length=self.training_args.max_length,
                return_tensors="pt"
            )
            return {
                "prompts": prompts,
                "input_ids": tokenized["input_ids"],
                "attention_mask": tokenized["attention_mask"],
                "true_answers": true_answers,
                "ids": [str(i) for i in range(len(batch))]
            }

        dataloader = DataLoader(
            self.formatted_test_dataset,
            batch_size=batch_size,
            collate_fn=collate_fn,
            shuffle=False
        )

        f1_scores = []

        for batch in tqdm(dataloader, desc=f"Evaluating F1 (batch_size={batch_size})"):
            input_ids = batch["input_ids"].to(self.model.device)
            attention_mask = batch["attention_mask"].to(self.model.device)

            generate_kwargs = {
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "max_new_tokens": 1024,
                "do_sample": False,
                "pad_token_id": self.tokenizer.eos_token_id,
                "eos_token_id": self.tokenizer.eos_token_id,
                "temperature": None,
                "top_p": None,
                "top_k": None,
            }

            with torch.no_grad():
                outputs = self.model.generate(**generate_kwargs)

            for i, (prompt, true_answer, example_id) in enumerate(zip(
                batch["prompts"], batch["true_answers"], batch["ids"]
            )):
                if true_answer is None:
                    continue

                prompt_ids = self.tokenizer(prompt, add_special_tokens=False)["input_ids"]
                prompt_len = len(prompt_ids)

                # Учитываем паддинг слева: количество добавленных pad-токенов
                pad_len = (attention_mask[i] == 0).sum().item()
                prompt_start = pad_len
                prompt_end = prompt_start + prompt_len

                generated_ids = outputs[i][prompt_end:]
                generated_part = self.tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

                pred_answer = self.extract_answer(generated_part)

                if pred_answer is None:
                    f1 = 0.0
                else:
                    pred_dict = {"prediction_text": pred_answer, "id": example_id}
                    ref_dict = {"answers": {"answer_start": [0], "text": [true_answer]}, "id": example_id}
                    result = self.squad_metric.compute(predictions=[pred_dict], references=[ref_dict])
                    f1 = result["f1"] / 100.0

                f1_scores.append(f1)

        self.tokenizer.padding_side = original_padding_side

        avg_f1 = np.mean(f1_scores) if f1_scores else 0.0
        self.metrics["f1_batch_2"] = round(float(avg_f1), 2)
        self.metrics["f1_list_batch_2"] = f1_scores


    @measure_metrics("evaluate_f1_batch_4")
    def evaluate_f1_batch_4(self, batch_size: int = 4):
        """
        Ба́тчевая версия evaluate_f1 для ускорения генерации.
        """
        original_padding_side = self.tokenizer.padding_side
        self.tokenizer.padding_side = "left"

        def collate_fn(batch):
            prompts = [item["prompt"] for item in batch]
            completions = [item["completion"] for item in batch]
            true_answers = [self.extract_answer(c) for c in completions]

            tokenized = self.tokenizer(
                prompts,
                padding=True,
                truncation=True,
                max_length=self.training_args.max_length,
                return_tensors="pt"
            )
            return {
                "prompts": prompts,
                "input_ids": tokenized["input_ids"],
                "attention_mask": tokenized["attention_mask"],
                "true_answers": true_answers,
                "ids": [str(i) for i in range(len(batch))]
            }

        dataloader = DataLoader(
            self.formatted_test_dataset,
            batch_size=batch_size,
            collate_fn=collate_fn,
            shuffle=False
        )

        f1_scores = []

        for batch in tqdm(dataloader, desc=f"Evaluating F1 (batch_size={batch_size})"):
            input_ids = batch["input_ids"].to(self.model.device)
            attention_mask = batch["attention_mask"].to(self.model.device)

            generate_kwargs = {
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "max_new_tokens": 1024,
                "do_sample": False,
                "pad_token_id": self.tokenizer.eos_token_id,
                "eos_token_id": self.tokenizer.eos_token_id,
                "temperature": None,
                "top_p": None,
                "top_k": None,
            }

            with torch.no_grad():
                outputs = self.model.generate(**generate_kwargs)

            for i, (prompt, true_answer, example_id) in enumerate(zip(
                batch["prompts"], batch["true_answers"], batch["ids"]
            )):
                if true_answer is None:
                    continue

                prompt_ids = self.tokenizer(prompt, add_special_tokens=False)["input_ids"]
                prompt_len = len(prompt_ids)

                # Учитываем паддинг слева: количество добавленных pad-токенов
                pad_len = (attention_mask[i] == 0).sum().item()
                prompt_start = pad_len
                prompt_end = prompt_start + prompt_len

                generated_ids = outputs[i][prompt_end:]
                generated_part = self.tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

                pred_answer = self.extract_answer(generated_part)

                if pred_answer is None:
                    f1 = 0.0
                else:
                    pred_dict = {"prediction_text": pred_answer, "id": example_id}
                    ref_dict = {"answers": {"answer_start": [0], "text": [true_answer]}, "id": example_id}
                    result = self.squad_metric.compute(predictions=[pred_dict], references=[ref_dict])
                    f1 = result["f1"] / 100.0

                f1_scores.append(f1)

        self.tokenizer.padding_side = original_padding_side

        avg_f1 = np.mean(f1_scores) if f1_scores else 0.0
        self.metrics["f1_batch_4"] = round(float(avg_f1), 2)
        self.metrics["f1_list_batch_4"] = f1_scores


    @measure_metrics("evaluate_f1_batch_8")
    def evaluate_f1_batch_8(self, batch_size: int = 8):
        """
        Ба́тчевая версия evaluate_f1 для ускорения генерации.
        """
        original_padding_side = self.tokenizer.padding_side
        self.tokenizer.padding_side = "left"

        def collate_fn(batch):
            prompts = [item["prompt"] for item in batch]
            completions = [item["completion"] for item in batch]
            true_answers = [self.extract_answer(c) for c in completions]

            tokenized = self.tokenizer(
                prompts,
                padding=True,
                truncation=True,
                max_length=self.training_args.max_length,
                return_tensors="pt"
            )
            return {
                "prompts": prompts,
                "input_ids": tokenized["input_ids"],
                "attention_mask": tokenized["attention_mask"],
                "true_answers": true_answers,
                "ids": [str(i) for i in range(len(batch))]
            }

        dataloader = DataLoader(
            self.formatted_test_dataset,
            batch_size=batch_size,
            collate_fn=collate_fn,
            shuffle=False
        )

        f1_scores = []

        for batch in tqdm(dataloader, desc=f"Evaluating F1 (batch_size={batch_size})"):
            input_ids = batch["input_ids"].to(self.model.device)
            attention_mask = batch["attention_mask"].to(self.model.device)

            generate_kwargs = {
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "max_new_tokens": 1024,
                "do_sample": False,
                "pad_token_id": self.tokenizer.eos_token_id,
                "eos_token_id": self.tokenizer.eos_token_id,
                "temperature": None,
                "top_p": None,
                "top_k": None,
            }

            with torch.no_grad():
                outputs = self.model.generate(**generate_kwargs)

            for i, (prompt, true_answer, example_id) in enumerate(zip(
                batch["prompts"], batch["true_answers"], batch["ids"]
            )):
                if true_answer is None:
                    continue

                prompt_ids = self.tokenizer(prompt, add_special_tokens=False)["input_ids"]
                prompt_len = len(prompt_ids)

                # Учитываем паддинг слева: количество добавленных pad-токенов
                pad_len = (attention_mask[i] == 0).sum().item()
                prompt_start = pad_len
                prompt_end = prompt_start + prompt_len

                generated_ids = outputs[i][prompt_end:]
                generated_part = self.tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

                pred_answer = self.extract_answer(generated_part)

                if pred_answer is None:
                    f1 = 0.0
                else:
                    pred_dict = {"prediction_text": pred_answer, "id": example_id}
                    ref_dict = {"answers": {"answer_start": [0], "text": [true_answer]}, "id": example_id}
                    result = self.squad_metric.compute(predictions=[pred_dict], references=[ref_dict])
                    f1 = result["f1"] / 100.0

                f1_scores.append(f1)

        self.tokenizer.padding_side = original_padding_side

        avg_f1 = np.mean(f1_scores) if f1_scores else 0.0
        self.metrics["f1_batch_8"] = round(float(avg_f1), 2)
        self.metrics["f1_list_batch_8"] = f1_scores



    @measure_metrics("evaluate_perplexity")
    def evaluate_perplexity(self):
        """
        Вычисляет перплексию на тестовом наборе без батчинга.
        Потери считаются только по completion.
        """
        self.model.eval()
        total_loss = 0.0
        total_tokens = 0

        for example in tqdm(self.formatted_test_dataset, desc="Calculating perplexity"):
            prompt = example["prompt"]
            completion = example["completion"]
            full_text = prompt + completion

            # Токенизируем полный текст (без паддинга, как одиночный пример)
            tokenized = self.tokenizer(
                full_text,
                truncation=True,
                max_length=self.training_args.max_length,
                return_tensors="pt"
            )
            input_ids = tokenized["input_ids"][0]  # убираем размерность батча
            # Токенизируем только промпт, чтобы узнать его длину
            prompt_ids = self.tokenizer(prompt, add_special_tokens=False)["input_ids"]
            prompt_len = len(prompt_ids)

            # Создаём labels: копируем input_ids, но маскируем промпт
            labels = input_ids.clone()
            labels[:prompt_len] = -100

            # Переносим на устройство модели
            input_ids = input_ids.unsqueeze(0).to(self.model.device)
            labels = labels.unsqueeze(0).to(self.model.device)
            attention_mask = torch.ones_like(input_ids).to(self.model.device)

            with torch.no_grad():
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss

            # Количество токенов completion (не -100)
            num_tokens = (labels != -100).sum().item()
            total_loss += loss.item() * num_tokens
            total_tokens += num_tokens

        avg_loss = total_loss / total_tokens if total_tokens > 0 else float("inf")
        perplexity = np.exp(avg_loss)

        self.metrics["perplexity"] = round(float(perplexity), 2)


    def run(self):
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        self.prepare_data()
        self.prepare_model()
        self.prepare_tokenizer()
        self.formatted_train_dataset = self.format_data(self.train_dataset)
        self.formatted_test_dataset = self.format_data(self.test_dataset)
        self.add_peft_if_exist()
        self.prepare_training()
        self.metrics[f"before_train_peak_memory_gb"] = round(torch.cuda.max_memory_allocated() / (1024**3), 2)
        self.train()
        self.model.eval()
        self.evaluate_f1()
        self.evaluate_f1_batch_2()
        self.evaluate_f1_batch_4()
        self.evaluate_f1_batch_8()
        self.evaluate_perplexity()

In [5]:
exp = RAGExperiment(
    dataset_name='phatvo/hotpotqa-raft',
    oracle_presence_ratio=1.0,
    sample_size=200,
    test_size_ratio=0.1,
    model_name='Qwen/Qwen2.5-0.5B-Instruct',
    peft_config=LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    ),
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=5,
    seed=1
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [6]:
exp.run()

README.md:   0%|          | 0.00/651 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/6.30M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1855 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1855 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1855 [00:00<?, ? examples/s]

...

Adding EOS to train dataset:   0%|          | 0/180 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/180 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,1.115669
20,1.050361
30,0.995238
40,0.939688
50,0.918599
60,0.869773


Calculating perplexity: 100%|██████████| 20/20 [00:01<00:00, 16.55it/s]


In [7]:
exp.metrics

{'trainable_params_m': 0.540672,
 'before_train_peak_memory_gb': 1.43,
 'train_time_min': 1.55,
 'train_peak_memory_gb': 9.87,
 'f1': 0.0,
 'evaluate_f1_time_min': 1.68,
 'evaluate_f1_peak_memory_gb': 1.0,
 'f1_batch_2': 0.0,
 'f1_list_batch_2': [0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0],
 'evaluate_f1_batch_2_time_min': 1.04,
 'evaluate_f1_batch_2_peak_memory_gb': 1.05,
 'f1_batch_4': 0.0,
 'f1_list_batch_4': [0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0],
 'evaluate_f1_batch_4_time_min': 0.74,
 'evaluate_f1_batch_4_peak_memory_gb': 1.17,
 'f1_batch_8': 0.0,
 'f1_list_batch_8': [0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0],
 'evaluate_f1_batch_8_time_min': 1.02,
 'evaluate_f1_batch_8_peak_memory_gb': 1.24,
 '

In [8]:
exp2 = RAGExperiment(
    dataset_name='phatvo/hotpotqa-raft',
    oracle_presence_ratio=1.0,
    sample_size=500,
    test_size_ratio=0.1,
    model_name='Qwen/Qwen2.5-0.5B-Instruct',
    peft_config=LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    ),
    num_train_epochs=10,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=5,
    seed=1
)

In [9]:
exp2.run()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/450 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/450 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,1.112376
20,1.050978
30,0.952531
40,0.927963
50,0.831259
60,0.788732
70,0.742107
80,0.720924
90,0.665876
100,0.652685


Calculating perplexity: 100%|██████████| 50/50 [00:03<00:00, 16.29it/s]


In [10]:
exp2.metrics

{'trainable_params_m': 0.540672,
 'before_train_peak_memory_gb': 2.38,
 'train_time_min': 12.59,
 'train_peak_memory_gb': 10.8,
 'f1': 0.59,
 'evaluate_f1_time_min': 6.45,
 'evaluate_f1_peak_memory_gb': 1.92,
 'f1_batch_2': 0.61,
 'f1_list_batch_2': [1.0,
  1.0,
  1.0,
  0.0,
  0.0,
  0.2857142857142857,
  1.0,
  1.0,
  0.0,
  1.0,
  0.0,
  1.0,
  0.4,
  1.0,
  1.0,
  0.3809523809523809,
  0.9,
  0.0,
  1.0,
  0.0,
  0.0,
  1.0,
  0.0,
  1.0,
  0.0,
  1.0,
  1.0,
  0.0,
  1.0,
  1.0,
  0.4,
  1.0,
  1.0,
  0.0,
  0.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  1.0,
  0.17391304347826086,
  0.0,
  1.0,
  0.0,
  0.0,
  0.14285714285714285,
  1.0],
 'evaluate_f1_batch_2_time_min': 3.94,
 'evaluate_f1_batch_2_peak_memory_gb': 1.97,
 'f1_batch_4': 0.54,
 'f1_list_batch_4': [1.0,
  1.0,
  1.0,
  0.0,
  0.0,
  0.2857142857142857,
  1.0,
  0.0,
  0.5,
  1.0,
  0.0,
  0.8571428571428571,
  0.4,
  1.0,
  1.0,
  0.09523809523809522,
  0.8571428571428571,
  0.0,
  1.0,
  0.0,
  0.0,
  1.0,